In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import mean_squared_error, r2_score
import statsmodels.api as sm
from scipy import stats

In [ ]:
# Load CSV data
data = pd.read_csv('/content/tips (data for regression problem).csv')

In [ ]:
# Data Analysis and Visualization
plt.figure(figsize=(20, 15))

In [ ]:
# 1. Scatter Plot: Total Bill vs Tip
plt.scatter(data['total_bill'], data['tip'])
plt.xlabel('Total Bill ($)')
plt.ylabel('Tip ($)')
plt.title('Scatter Plot: Total Bill vs Tip')

In [ ]:
 #2. Pair Plot
numeric_cols = ['total_bill', 'tip', 'size']
sns.pairplot(data[numeric_cols])
plt.title('Pair Plot for Numeric Features')

In [ ]:
# 3. Correlation Analysis
# Convert categorical variables to numeric
categorical_cols = ['sex', 'smoker', 'day', 'time']
data_encoded = data.copy()
le = LabelEncoder()
for col in categorical_cols:
    data_encoded[col] = le.fit_transform(data[col])

In [ ]:
# Correlation Matrix
correlation = data_encoded.corr()
sns.heatmap(correlation, annot=True, cmap='coolwarm')
plt.title('Correlation Matrix Heatmap')

In [ ]:
# 4. Residuals Plot
X = data['total_bill'].values.reshape(-1, 1)
y = data['tip'].values
model = LinearRegression()
model.fit(X, y)
y_pred = model.predict(X)
residuals = y - y_pred

In [ ]:
plt.scatter(y_pred, residuals)
plt.axhline(y=0, color='r', linestyle='--')
plt.xlabel('Predicted Tips ($)')
plt.ylabel('Residuals')
plt.title('Residuals Plot')

In [ ]:
# 5. Time Series Plot
plt.plot(range(len(data)), data['tip'], marker='o')
plt.xlabel('Order Index')
plt.ylabel('Tip Amount ($)')
plt.title('Time Series of Tips')

In [ ]:
# Rainbow Test for Linearity
def rainbow_test(X, y):
    X = sm.add_constant(X)
    model = sm.OLS(y, X).fit()
    rainbow_statistic, rainbow_p_value = sm.stats.diagnostic.linear_rainbow(model)
    return rainbow_statistic, rainbow_p_value


In [ ]:
X = data_encoded.drop(['tip'], axis=1)
y = data_encoded['tip']

In [ ]:
# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
models = {
    'Linear Regression': LinearRegression(),
    'Ridge': Ridge(alpha=1.0),
    'Lasso': Lasso(alpha=1.0),
    'Decision Tree': DecisionTreeRegressor(random_state=42),
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42),
    'SVR': SVR(kernel='rbf'),
    'KNN': KNeighborsRegressor(n_neighbors=3)
}

In [ ]:
results = {}
for name, model in models.items():
    # Train model
    model.fit(X_train_scaled, y_train)

    # Make predictions
    y_pred = model.predict(X_test_scaled)

    # Calculate metrics
    mse = mean_squared_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)

    # Cross-validation
    cv_scores = cross_val_score(model, X_train_scaled, y_train, cv=3, scoring='r2')

    results[name] = {
        'MSE': mse,
        'R2': r2,
        'CV_R2_Mean': cv_scores.mean(),
        'CV_R2_Std': cv_scores.std()
    }

In [ ]:
# Feature importance analysis using Random Forest
rf_model = models['Random Forest']
feature_importance = pd.DataFrame({
    'Feature': X.columns,
    'Importance': rf_model.feature_importances_
}).sort_values('Importance', ascending=False)

In [ ]:
print("\n=== Model Performance Results ===")
for name, metrics in results.items():
    print(f"\n{name}:")
    print(f"MSE: {metrics['MSE']:.4f}")
    print(f"R2 Score: {metrics['R2']:.4f}")
    print(f"Cross-validation R2 (mean ? std): {metrics['CV_R2_Mean']:.4f} ? {metrics['CV_R2_Std']:.4f}")

print("\n=== Feature Importance ===")
print(feature_importance)

In [ ]:
rainbow_stat, rainbow_p = rainbow_test(X['total_bill'].values.reshape(-1, 1), y)
print(f"\n=== Rainbow Test Results ===")
print(f"Statistic: {rainbow_stat:.4f}")
print(f"P-value: {rainbow_p:.4f}")

plt.tight_layout()
plt.show()

In [ ]:
print("\n=== Management Insights ===")
print("\n1. Average Tips by Party Size:")
print(data.groupby('size')['tip'].agg(['mean', 'count']))

In [ ]:
print("\n2. Tips by Gender:")
print(data.groupby('sex')['tip'].agg(['mean', 'count']))

In [ ]:
print("\n3. Average Tip Percentage:")
data['tip_percentage'] = (data['tip'] / data['total_bill']) * 100
print(f"Average tip percentage: {data['tip_percentage'].mean():.2f}%")

In [ ]:
with open('tip_analysis_results.txt', 'w') as f:
    f.write("=== Tipping Behavior Analysis Results ===\n\n")
    f.write("1. Key Factors Influencing Tips:\n")
    f.write(feature_importance.to_string())
    f.write("\n\n2. Model Performance Summary:\n")
    for name, metrics in results.items():
        f.write(f"\n{name}:\n")
        f.write(f"R2 Score: {metrics['R2']:.4f}\n")
    f.write("\n3. Management Recommendations:\n")
    f.write("- Focus on larger party bookings\n")
    f.write("- Monitor service quality for high-bill tables\n")
    f.write("- Consider special attention to weekend dinner service\n")